# U2-01 · Estadística descriptiva y valores atípicos

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 2, apartados 2.2 y 2.4*

Reproduce los ejemplos 2.1 y 2.3 del libro con los diez tiempos de ciclo de una máquina (datos ficticios).

Ejecuta cada celda con **Mayús + Intro**. Puedes cambiar los datos y volver a ejecutar.

## 1. Los datos

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

tiempos = np.array([41.2, 39.8, 40.5, 42.1, 40.9, 39.5, 41.7, 40.3, 51.0, 40.1])  # segundos
np.sort(tiempos)

## 2. Posición y dispersión

In [ ]:
n = len(tiempos)
media, mediana = tiempos.mean(), np.median(tiempos)
s = tiempos.std(ddof=1)            # desviación típica muestral (divide entre n - 1)
q1, q3 = np.percentile(tiempos, [25, 75])
ric = q3 - q1
print(f'Media {media:.2f} s · Mediana {mediana:.2f} s')
print(f'Desviación típica {s:.2f} s · Rango {tiempos.max() - tiempos.min():.1f} s · CV {s / media:.3f}')
print(f'Q1 {q1:.2f} · Q3 {q3:.2f} · RIC {ric:.2f}')

Prueba a quitar el ciclo de 51,0 s: ¿qué medidas cambian mucho y cuáles casi nada?

In [ ]:
sin_atipico = tiempos[tiempos < 50]
print(f'Media {sin_atipico.mean():.2f} · Mediana {np.median(sin_atipico):.2f} · s {sin_atipico.std(ddof=1):.2f}')

## 3. Diagrama de caja e histograma

In [ ]:
import matplotlib.pyplot as plt
fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3))
a.boxplot(tiempos, whis=1.5)
a.set_ylabel('Tiempo de ciclo (s)'); a.set_xticks([])
b.hist(tiempos, bins=8, color='#1C8C96')
b.set_xlabel('Tiempo de ciclo (s)'); b.set_ylabel('Frecuencia')
plt.tight_layout(); plt.show()

## 4. Asimetría: dos fórmulas, dos resultados

`scipy` usa por defecto la fórmula sin corrección y `pandas` la corregida (Joanes y Gill, 1998).

In [ ]:
print('scipy  (sin corrección):', round(stats.skew(tiempos), 2))
print('pandas (corregida):    ', round(pd.Series(tiempos).skew(), 2))
print('Curtosis en exceso (pandas):', round(pd.Series(tiempos).kurt(), 2))

## 5. Tres reglas para detectar atípicos

In [ ]:
def regla_desviacion(x, k=3):
    m, s = x.mean(), x.std(ddof=1)
    return m - k * s, m + k * s

def regla_tukey(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    return q1 - k * (q3 - q1), q3 + k * (q3 - q1)

def regla_mad(x, k=2.5, b=1.4826):           # Leys et al. (2013)
    med = np.median(x)
    mad = b * np.median(np.abs(x - med))
    return med - k * mad, med + k * mad

for nombre, regla in [('Media ± 3 s', regla_desviacion), ('Vallas de Tukey', regla_tukey), ('Mediana ± 2,5 MAD', regla_mad)]:
    lo, hi = regla(tiempos)
    fuera = tiempos[(tiempos < lo) | (tiempos > hi)]
    print(f'{nombre:18s} límites [{lo:6.2f}, {hi:6.2f}]  atípicos: {fuera}')

## 6. Por qué falla la primera regla

Con $n$ datos, ningún valor puede alejarse de la media más de $(n-1)/\sqrt{n}$ desviaciones típicas.

In [ ]:
for n in [5, 10, 12, 20, 50]:
    print(n, round((n - 1) / np.sqrt(n), 2))

Con diez datos el máximo es 2,85: la regla de las tres desviaciones típicas no puede marcar ningún valor, por extremo que sea.

> Leys, C. et al. (2013). Detecting outliers. *Journal of Experimental Social Psychology*, 49(4), 764–766. https://doi.org/10.1016/j.jesp.2013.03.013